In [ ]:
%%markdown
# Parameter Counting for Pruned Models

This notebook scans the unstructured-pruned models under `models/pruned-models-v2/`
(`magnitude` / `wanda` / `sparsegpt`, stored as `model.safetensors`) and reports, for every model:

- **Total** parameters stored in the checkpoint.
- **Zero** parameters — weights stored as exactly `0.0` (what unstructured pruning leaves behind).
- **Real** parameters — non-zero count = `total - zeros` (what the model effectively carries).
- **Sparsity %** — `zeros / total`.

Each metric is reported in **two scopes**:

1. **All tensors** — true overall stored sparsity / real total.
2. **Prunable weights only** — the 2-D projection weight matrices that pruning actually targets
   (`q/k/v/o_proj`, `gate/up/down_proj`), excluding embeddings, `lm_head`, norms, and biases.
   This reflects the real sparsity of the layers pruning operates on, so it should match the
   nominal sparsity level (30 / 50 / 70) in each model's `recipe.yaml`.

Because unstructured pruning zeros weights in place, the **total** is identical across sparsity
levels — the zero / real / sparsity columns are what reveal the effect of pruning.

Missing packages (`safetensors`, `numpy`) are installed automatically.


In [ ]:
import sys
import subprocess
from pathlib import Path
from math import prod


def _ensure(packages):
    """Import-or-install helper for the notebook environment."""
    missing = []
    for mod, pip_name in packages:
        try:
            __import__(mod)
        except ImportError:
            missing.append(pip_name)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


_ensure([("safetensors", "safetensors"), ("numpy", "numpy")])

import numpy as np
from safetensors import safe_open

# ----------------------------------------------------------------------------
# Configuration -- edit this to point at a different model collection.
MODELS_DIR = Path("/home/orin/toy-gace/inference-benchmark/models/pruned-models-v2")
LLM_PRUNER_SUBDIR = "llm-pruner-models"  # structured models, skipped here
MODEL_FILENAME = "model.safetensors"
# ----------------------------------------------------------------------------

# Weight matrices that pruning actually targets (see each model's recipe.yaml).
PROJ = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def is_prunable(name: str, ndim: int) -> bool:
    """A prunable weight = a 2-D projection matrix (excludes embeddings/lm_head/norms/biases)."""
    return ndim == 2 and any(p in name for p in PROJ)


def analyze_safetensors(path: Path) -> dict:
    """Count total and zero parameters in a safetensors file (all tensors + prunable-only)."""
    counters = dict(total_all=0, zeros_all=0, total_prune=0, zeros_prune=0)
    with safe_open(str(path), framework="np", device="cpu") as f:
        for name in f.keys():
            tensor = f.get_tensor(name)
            total = int(prod(tensor.shape))
            zeros = total - int(np.count_nonzero(tensor))
            counters["total_all"] += total
            counters["zeros_all"] += zeros
            if is_prunable(name, tensor.ndim):
                counters["total_prune"] += total
                counters["zeros_prune"] += zeros
    return counters


def classify(name: str) -> str:
    """Human-readable pruning-method label."""
    for method in ("magnitude", "wanda", "sparsegpt"):
        if method in name:
            return method
    return "unknown"


def make_row(name: str, counters: dict) -> dict:
    total_all = counters["total_all"]
    total_prune = counters["total_prune"]
    return {
        "model": name,
        "method": classify(name),
        "total": total_all,
        "zeros_all": counters["zeros_all"],
        "real_all": total_all - counters["zeros_all"],
        "sparsity_all_%": (counters["zeros_all"] / total_all * 100) if total_all else 0.0,
        "prunable_total": total_prune,
        "prunable_zeros": counters["zeros_prune"],
        "prunable_real": total_prune - counters["zeros_prune"],
        "prunable_sparsity_%": (counters["zeros_prune"] / total_prune * 100) if total_prune else 0.0,
    }


# Discover unstructured (safetensors) models, skipping the structured llm-pruner folder.
model_dirs = sorted(
    p for p in MODELS_DIR.iterdir()
    if p.is_dir() and p.name != LLM_PRUNER_SUBDIR and (p / MODEL_FILENAME).exists()
)

rows = []
errors = []
for model_dir in model_dirs:
    try:
        rows.append(make_row(model_dir.name, analyze_safetensors(model_dir / MODEL_FILENAME)))
    except Exception as exc:
        errors.append((model_dir.name, repr(exc)))

# Display results.
if not rows:
    print(f"No models containing {MODEL_FILENAME} found under {MODELS_DIR}.")
else:
    try:
        import pandas as pd

        df = pd.DataFrame(rows)
        int_cols = ["total", "zeros_all", "real_all", "prunable_total", "prunable_zeros", "prunable_real"]
        fmt = {c: "{:,}".format for c in int_cols}
        fmt["sparsity_all_%"] = "{:.1f}".format
        fmt["prunable_sparsity_%"] = "{:.1f}".format
        with pd.option_context("display.max_rows", None, "display.width", None,
                               "display.max_colwidth", None):
            try:
                display(df.style.format(fmt))  # noqa: F821 (Jupyter builtin)
            except NameError:
                print(df.to_string(index=False, formatters=fmt))
    except ImportError:
        header = ["Model", "Method", "Total", "Zeros", "Real", "Sparsity%",
                  "PrunTotal", "PrunZeros", "PrunReal", "PrunSparsity%"]
        row_format = "{:<58} {:<10} {:>14} {:>14} {:>14} {:>9} {:>14} {:>14} {:>14} {:>13}"
        print(row_format.format(*header))
        print("-" * 190)
        for r in rows:
            print(row_format.format(
                r["model"], r["method"],
                f'{r["total"]:,}', f'{r["zeros_all"]:,}', f'{r["real_all"]:,}', f'{r["sparsity_all_%"]:.1f}',
                f'{r["prunable_total"]:,}', f'{r["prunable_zeros"]:,}', f'{r["prunable_real"]:,}',
                f'{r["prunable_sparsity_%"]:.1f}',
            ))

    print(f"\nScanned {len(rows)} unstructured models under {MODELS_DIR}.")
    if errors:
        print(f"\n{len(errors)} model(s) failed:")
        for name, err in errors:
            print(f"  {name}: {err}")
